# 05-07 RunnableWithMessageHistory: 대화 기록 관리하기

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있어요:

- `RunnableWithMessageHistory`로 체인에 세션별 대화 기록을 추가할 수 있어요
- `MessagesPlaceholder`의 역할을 이해하고, `history_messages_key`와 변수명을 올바르게 연결할 수 있어요
- 같은 세션 내에서 이전 대화 맥락이 유지되고, 다른 세션과 대화가 섞이지 않는 것을 확인할 수 있어요
- `ConfigurableFieldSpec`으로 사용자 ID와 대화 ID를 조합한 세션 키를 설계할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 다음 개념을 알고 있으면 좋아요:

- `ChatPromptTemplate`과 메시지 역할(system/human/ai) 이해 (Ch02)
- LCEL 파이프 연산자(`|`) 사용법 (Ch05-01~03)
- Ch04 Memory 기본 개념 (ConversationBufferMemory 등)
- Ch05-06 Configure에서 배운 `ConfigurableField` 패턴

---

`RunnableWithMessageHistory`는 LangChain의 공식 대화 기록(Message History) 관리 방식이에요. Ch04에서 배운 `ConversationBufferMemory` 같은 Memory 클래스를 대체하는 LCEL 기반 접근법이에요.

> 🔑 **핵심 개념**: `RunnableWithMessageHistory`는 Ch04의 Memory 클래스를 대체하는 LangChain의 공식 대화 기록 관리 방식이에요. Memory 클래스가 체인 외부에서 상태를 관리했다면, `RunnableWithMessageHistory`는 LCEL 파이프라인 안에서 대화 기록을 자연스럽게 주입해요.

**주요 특징:**
- 세션(Session) ID 기반으로 대화 기록을 분리 관리해요
- 이전 대화 맥락을 유지하여 연속적인 대화가 가능해요
- 인메모리(In-Memory) 또는 영구 저장소(Redis, PostgreSQL 등)를 선택할 수 있어요

아래 다이어그램은 `RunnableWithMessageHistory`가 대화 기록을 어떻게 관리하는지 보여줘요.

```mermaid
sequenceDiagram
    participant U as 사용자
    participant R as RunnableWithMessageHistory
    participant S as 세션 저장소<br/>(store 딕셔너리)
    participant P as 프롬프트<br/>(MessagesPlaceholder)
    participant L as LLM

    U->>R: invoke(input, config={session_id})
    R->>S: get_session_history(session_id)
    S-->>R: 이전 대화 기록 반환
    R->>P: 대화 기록 + 현재 입력 조립
    P->>L: 완성된 프롬프트 전달
    L-->>R: 응답 반환
    R->>S: 현재 입력 + 응답을 기록에 추가
    R-->>U: 응답 전달
```

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


True

## 기본 체인 준비

먼저 대화 기록을 삽입할 수 있는 프롬프트와 기본 체인을 만들어요.

핵심은 `MessagesPlaceholder(variable_name="history")`에요. 이것은 프롬프트 안에 **대화 기록이 삽입될 자리**를 미리 비워두는 자리표시자(Placeholder)예요. 나중에 `RunnableWithMessageHistory`가 이 자리에 세션별 대화 메시지를 자동으로 채워 넣어요.

```
프롬프트 구조:
┌──────────────────────────────────────┐
│ SystemMessage: "당신은 ... 어시스턴트" │
│ MessagesPlaceholder: [이전 대화 기록]  │  ← 여기에 히스토리가 삽입돼요
│ HumanMessage: "현재 질문"             │
└──────────────────────────────────────┘
```

In [ ]:
# ============================================================
# TODO: MessagesPlaceholder를 포함한 프롬프트와 기본 체인을 완성하세요
# 힌트: MessagesPlaceholder(variable_name="history") — 대화 기록이 삽입될 위치
# 예상 결과: "✅ 기본 체인 생성 완료" 출력
# ============================================================

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_openai import ChatOpenAI

# ---------------------------------------------------
# MessagesPlaceholder: 대화 기록을 프롬프트에 삽입하는 자리표시자
# ---------------------------------------------------

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")

# 대화 기록을 포함한 프롬프트 템플릿
# MessagesPlaceholder: 이 위치에 세션별 대화 기록이 순서대로 삽입됨
# - variable_name 값은 RunnableWithMessageHistory의 history_messages_key와 반드시 일치해야 함
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 {ability}에 능숙한 어시스턴트입니다. 20자 이내로 응답하세요.",
        ),
        # MessagesPlaceholder: 이전 대화 메시지들이 순서대로 삽입됨
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),  # 현재 사용자 입력
    ]
)

# 기본 체인 생성 (히스토리 없이 — 다음 단계에서 RunnableWithMessageHistory로 감쌈)
runnable = prompt | model

print("=" * 60)
print("✅ 기본 체인 생성 완료")
print("=" * 60)


### MessagesPlaceholder 이해하기

`MessagesPlaceholder`는 대화 기록처럼 **여러 메시지를 한 번에 삽입**할 때 사용하는 LangChain 구성 요소예요. `RunnableWithMessageHistory`와 함께 쓰면, 세션별로 누적된 메시지를 프롬프트 안에 자연스럽게 끼워넣을 수 있어요.

### 왜 필요할까요?

- 체인이 실행될 때마다 직전에 오간 대화를 **전달**해야 LLM이 맥락을 기억해요
- `MessagesPlaceholder(variable_name="history")`는 해당 변수 이름으로 전달된 메시지 목록을 그 자리에 삽입해요
- `RunnableWithMessageHistory`에서 `history_messages_key="history"`로 지정하면, 프롬프트의 `MessagesPlaceholder`와 연결돼요

### 이름 일치 규칙

```
MessagesPlaceholder(variable_name="history")
                                      ↕ 반드시 일치
RunnableWithMessageHistory(..., history_messages_key="history")
```

> ⚠️ **주의**: `variable_name`과 `history_messages_key`가 다르면 대화 기록이 프롬프트에 전달되지 않아요. 두 값을 항상 일치시키세요.

## 1. 인메모리 대화 기록 (In-Memory)

가장 간단한 방법으로, 대화 기록을 파이썬 딕셔너리에 저장해요.

**동작 원리:**
1. `store` 딕셔너리가 `{session_id: ChatMessageHistory}` 형태로 세션별 대화 기록을 보관해요
2. `get_session_history(session_id)` 함수가 세션 ID에 해당하는 기록을 반환해요
3. `RunnableWithMessageHistory`가 매 호출마다 이 함수를 사용하여 대화 기록을 가져오고, 새 메시지를 추가해요

**인메모리 방식의 특징:**
- 빠른 접근 속도
- 애플리케이션 재시작 시 기록 소실
- 개발/테스트 환경에 적합

> 🎯 **강의 포인트**: `session_id`가 세션을 구분하는 유일한 키예요. 같은 `session_id`면 대화가 이어지고, 다른 `session_id`면 새 대화가 시작돼요. 이 원리를 먼저 설명하면 이후 코드가 훨씬 이해하기 쉬워요.

> ⚠️ **주의**: `MessagesPlaceholder(variable_name="history")`의 `variable_name` 값과 `RunnableWithMessageHistory`의 `history_messages_key` 값이 반드시 일치해야 해요. 두 값이 다르면 대화 기록이 프롬프트에 전달되지 않아요.

In [ ]:
# ============================================================
# TODO: get_session_history 함수를 작성하고 RunnableWithMessageHistory를 생성하세요
# 힌트: session_id를 키로 store 딕셔너리에서 ChatMessageHistory 반환
#       RunnableWithMessageHistory(runnable, get_session_history, input_messages_key="input", history_messages_key="history")
# 예상 결과: 세션별 독립 대화 기록 관리
# ============================================================

# ---------------------------------------------------
# RunnableWithMessageHistory: 체인에 세션별 대화 기록 추가
# ---------------------------------------------------

# 1단계: 세션 기록을 저장할 딕셔너리
# {session_id: ChatMessageHistory} 형태로 세션별 대화 기록 보관
store = {}


# 2단계: 세션 ID를 기반으로 세션 기록을 가져오는 함수
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """세션 ID에 해당하는 대화 기록을 반환"""
    if session_id not in store:
        # 세션이 없으면 새로 생성 (ChatMessageHistory: 인메모리 대화 기록 저장소)
        store[session_id] = ChatMessageHistory()
    return store[session_id]


# 3단계: RunnableWithMessageHistory 생성
# 주요 파라미터:
# - runnable: 실행할 기본 체인
# - get_session_history: 세션 ID → ChatMessageHistory 반환 함수
# - input_messages_key: 현재 사용자 입력의 딕셔너리 키
# 💡 팁: history_messages_key 값은 MessagesPlaceholder의 variable_name과 반드시 같아야 합니다
with_message_history = RunnableWithMessageHistory(
    runnable,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",  # MessagesPlaceholder(variable_name="history")와 일치
)

print("=" * 60)
print("✅ RunnableWithMessageHistory 생성 완료")
print("=" * 60)
print("이제 세션별로 대화 기록이 관리됩니다.")

### 1.1 같은 세션에서 대화 이어가기

아래에서 세 가지를 확인해 볼게요:

1. **같은 session_id** (`"user123"`)로 두 번 대화하면 이전 맥락이 유지되는지
2. **다른 session_id** (`"user456"`)로 대화하면 이전 대화를 모르는지
3. 세션별로 대화 기록이 완전히 분리되는지

> ⚠️ **주의**: `session_id`가 다르면 대화 기록이 분리돼요. 같은 세션을 공유하려면 동일한 ID를 사용하세요.

In [4]:
# 첫 번째 대화
# 
# 중요: config에 session_id를 반드시 지정해야 합니다
result1 = with_message_history.invoke(
    {"ability": "수학", "input": "코사인의 의미는 무엇인가요?"},
    config={"configurable": {"session_id": "user123"}},
)

print("=" * 60)
print("📥 첫 번째 대화")
print("=" * 60)
print(f"질문: 코사인의 의미는 무엇인가요?")
print()
print("📤 답변:")
print(result1.content)


📥 첫 번째 대화
질문: 코사인의 의미는 무엇인가요?

📤 답변:
코사인은 각의 인접변과 빗변의 비율입니다.


In [5]:
# 두 번째 대화 (같은 세션)
# 
# 이전 대화 내용을 기억하고 연속적으로 대화 가능
result2 = with_message_history.invoke(
    {"ability": "수학", "input": "이전의 내용을 한글로 답변해 주세요."},
    config={"configurable": {"session_id": "user123"}},
)

print("=" * 60)
print("📥 두 번째 대화 (같은 세션)")
print("=" * 60)
print(f"질문: 이전의 내용을 한글로 답변해 주세요.")
print()
print("📤 답변:")
print(result2.content)
print()
print("💡 이전 대화 내용을 기억하여 연속적인 대화가 가능합니다!")


📥 두 번째 대화 (같은 세션)
질문: 이전의 내용을 한글로 답변해 주세요.

📤 답변:
코사인은 각의 인접변과 빗변의 비율입니다.

💡 이전 대화 내용을 기억하여 연속적인 대화가 가능합니다!


### 1.2 다른 세션으로 대화 — 세션 격리(Session Isolation) 확인

이번에는 `session_id`를 `"user456"`으로 바꿔서 대화해 볼게요. 이 세션에는 이전 대화 기록이 없으므로, "이전의 내용을 한글로 답변해 주세요"라고 해도 참조할 맥락이 없어요.

In [6]:
# 다른 세션으로 대화
# 
# 다른 session_id를 사용하면 새로운 대화가 시작됨
result3 = with_message_history.invoke(
    {"ability": "수학", "input": "이전의 내용을 한글로 답변해 주세요."},
    config={"configurable": {"session_id": "user456"}},
)

print("=" * 60)
print("📥 다른 세션으로 대화")
print("=" * 60)
print(f"질문: 이전의 내용을 한글로 답변해 주세요.")
print()
print("📤 답변:")
print(result3.content)
print()
print("💡 다른 세션이므로 이전 대화 내용을 기억하지 못함")


📥 다른 세션으로 대화
질문: 이전의 내용을 한글로 답변해 주세요.

📤 답변:
20자 이내로 요청하세요.

💡 다른 세션이므로 이전 대화 내용을 기억하지 못함


## 2. 사용자 정의 세션 키 — 멀티 유저 패턴

기본 `session_id` 하나로는 "같은 사용자의 여러 대화"를 구분하기 어려워요. `ConfigurableFieldSpec`을 사용하면 `user_id`와 `conversation_id`를 조합한 복합 세션 키를 만들 수 있어요.

### 언제 필요할까요?

| 시나리오 | 기본 session_id | 복합 키 (user_id + conversation_id) |
|----------|-----------------|-------------------------------------|
| 1명이 1개 대화 | 충분해요 | 불필요 |
| 1명이 여러 대화 | 구분 불가 | `user_id`로 사용자, `conversation_id`로 대화방 구분 |
| 여러 명이 여러 대화 | 구분 불가 | 사용자별 + 대화별 완전 격리 |

> 🎯 **강의 포인트**: `(user_id, conversation_id)` 튜플을 키로 사용하는 패턴은 실제 서비스에서 "한 유저가 여러 대화를 동시에 관리"하는 시나리오를 그대로 구현해요. 카카오톡의 채팅방 개념으로 비유하면 이해가 쉬워요.

> 💡 **실무 팁**: 프로덕션 환경에서는 인메모리 저장소(`ChatMessageHistory`) 대신 `RedisChatMessageHistory`를 사용하는 것을 권장해요. 서버가 재시작되어도 대화 기록이 유지되고, 여러 서버 인스턴스 간에 세션을 공유할 수 있어요.

In [ ]:
from langchain_core.runnables import ConfigurableFieldSpec

# ============================================================
# TODO: user_id와 conversation_id를 조합하는 사용자 정의 세션 키를 설정하세요
# 힌트: ConfigurableFieldSpec(id="user_id", ...) + ConfigurableFieldSpec(id="conversation_id", ...)
# 예상 결과: config={"configurable": {"user_id": "...", "conversation_id": "..."}}으로 세션 구분
# ============================================================

# ---------------------------------------------------
# ConfigurableFieldSpec: 사용자 정의 세션 키 설계
# ---------------------------------------------------

# 1단계: 사용자 정의 세션 키를 사용한 저장소
# (user_id, conversation_id) 튜플을 키로 사용하여 더 세밀한 세션 관리
custom_store = {}


def get_custom_session_history(user_id: str, conversation_id: str) -> BaseChatMessageHistory:
    """user_id와 conversation_id를 조합하여 세션 관리"""
    key = (user_id, conversation_id)
    if key not in custom_store:
        custom_store[key] = ChatMessageHistory()
    return custom_store[key]


# 2단계: 사용자 정의 세션 키로 RunnableWithMessageHistory 생성
# ConfigurableFieldSpec: 세션 키 필드 정의
# - id: config에서 사용할 키 이름 (get_custom_session_history 인자명과 일치해야 함)
# - annotation: 타입 힌트
# - is_shared: True이면 여러 Runnable에서 공유
with_custom_history = RunnableWithMessageHistory(
    runnable,
    get_custom_session_history,
    input_messages_key="input",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="user_id",
            annotation=str,
            name="User ID",
            description="사용자의 고유 식별자입니다.",
            default="",
            is_shared=True,
        ),
        ConfigurableFieldSpec(
            id="conversation_id",
            annotation=str,
            name="Conversation ID",
            description="대화의 고유 식별자입니다.",
            default="",
            is_shared=True,
        ),
    ],
)

print("=" * 60)
print("✅ 사용자 정의 세션 키 설정 완료")
print("=" * 60)


In [8]:
# 사용자 정의 세션 키로 대화
result = with_custom_history.invoke(
    {"ability": "수학", "input": "코사인의 의미는 무엇인가요?"},
    config={"configurable": {"user_id": "user123", "conversation_id": "conv001"}},
)

print("=" * 60)
print("📥 사용자 정의 세션 키로 대화")
print("=" * 60)
print(result.content)


📥 사용자 정의 세션 키로 대화
코사인은 각의 인접변과 빗변의 비율입니다.


## 3. 영구 저장소 연동 (Redis 예시)

지금까지 사용한 인메모리 방식은 프로그램이 종료되면 대화 기록이 사라져요. 프로덕션 환경에서는 Redis, PostgreSQL 같은 영구 저장소를 사용하여 대화 기록을 보존해요.

아래는 Redis를 사용하는 예시 코드예요. Redis 서버가 필요하므로 주석 처리되어 있어요.

> 💡 **실무 팁**: 저장소만 바꿔도 나머지 코드는 동일해요. `get_session_history` 함수가 `ChatMessageHistory` 대신 `RedisChatMessageHistory`를 반환하도록 변경하면 돼요. 이것이 추상화(Abstraction)의 장점이에요.

In [9]:
# Redis를 사용한 대화 기록 (선택사항)
# 
# 주의: Redis 서버가 실행 중이어야 합니다
# 
# Redis 사용 예제:
# from langchain_community.chat_message_histories import RedisChatMessageHistory
# 
# REDIS_URL = "redis://localhost:6379/0"
# 
# def get_redis_history(session_id: str) -> RedisChatMessageHistory:
#     return RedisChatMessageHistory(session_id, url=REDIS_URL)
# 
# with_redis_history = RunnableWithMessageHistory(
#     runnable,
#     get_redis_history,
#     input_messages_key="input",
#     history_messages_key="history",
# )

print("=" * 60)
print("📝 Redis 사용 예제 (주석 처리됨)")
print("=" * 60)
print("Redis를 사용하려면:")
print("1. pip install redis")
print("2. Redis 서버 실행")
print("3. 위 주석을 해제하고 사용")


📝 Redis 사용 예제 (주석 처리됨)
Redis를 사용하려면:
1. pip install redis
2. Redis 서버 실행
3. 위 주석을 해제하고 사용


## 정리

### RunnableWithMessageHistory 주요 파라미터

| 파라미터 | 역할 | 예시 |
|----------|------|------|
| `runnable` | 실행할 기본 체인 | `prompt \| model` |
| `get_session_history` | 세션 ID로 기록 반환하는 함수 | `lambda sid: store[sid]` |
| `input_messages_key` | 현재 사용자 입력의 딕셔너리 키 | `"input"` |
| `history_messages_key` | 대화 기록의 딕셔너리 키 | `"history"` |
| `history_factory_config` | 사용자 정의 세션 키 설정 | `[ConfigurableFieldSpec(...)]` |

### 이름 일치 체크리스트

설정을 연결할 때 아래 이름들이 올바르게 일치하는지 확인하세요:

```
MessagesPlaceholder(variable_name="history")     ← 프롬프트 측
                                    ↕ 일치
RunnableWithMessageHistory(..., history_messages_key="history")  ← 히스토리 측
```

```
get_session_history(user_id, conversation_id)     ← 함수 인자명
                     ↕ 일치              ↕ 일치
ConfigurableFieldSpec(id="user_id")  ConfigurableFieldSpec(id="conversation_id")
```

### 이번 노트북에서 배운 핵심

1. **RunnableWithMessageHistory**: LCEL 파이프라인에 세션별 대화 기록을 자동으로 추가해요
2. **MessagesPlaceholder**: 프롬프트 안에 대화 기록이 삽입될 자리를 만들어요
3. **세션 격리**: `session_id`가 다르면 대화 기록이 완전히 분리돼요
4. **복합 세션 키**: `ConfigurableFieldSpec`으로 user_id + conversation_id 조합이 가능해요
5. **저장소 교체**: `get_session_history` 함수만 바꾸면 인메모리에서 Redis 등으로 전환할 수 있어요

### Ch04 Memory vs RunnableWithMessageHistory 비교

| 항목 | Ch04 Memory (구 방식) | RunnableWithMessageHistory (신 방식) |
|------|----------------------|--------------------------------------|
| 방식 | 체인 외부에서 상태 관리 | LCEL 파이프라인 내부에서 관리 |
| 세션 관리 | 수동 구현 필요 | `session_id` 기반 자동 관리 |
| 저장소 | Memory 클래스별로 다름 | `get_session_history` 함수로 통일 |
| LCEL 호환 | 제한적 | 완전 호환 |

### 다음 노트북 예고

다음 노트북(`08-Custom-Generator`)에서는 파이썬 제너레이터(Generator) 함수를 LCEL 파이프라인에 연결하여 스트리밍(Streaming) 방식으로 데이터를 실시간 처리하는 방법을 배워요.